<div style="border-left: 8px solid #04B4E3; padding: 0.25rem 0 0.25rem 1rem;">
<h1 style="color: #133C5A; margin-bottom: 0.25rem;">Expansão da Rede Federal de Educação Profissional e Economia Municipal</h1>
<p style="color: #00598E; margin: 0;"><strong>Notebook acadêmico principal</strong> · 2007–2019 · Município-ano</p>
</div>

**Pergunta de pesquisa.** A entrada em operação de unidades associadas à Fase II da expansão da Rede Federal alterou a atividade econômica dos municípios?

Este notebook apresenta a evidência construída até aqui. O período é 2007–2019, a unidade analítica é município-ano e o outcome econômico principal futuro é o pessoal ocupado assalariado (CEMPRE 708). O objetivo causal segue em avaliação; este documento não estima efeito causal.

## 1. Motivação e desenho

Na Fase II, municípios receberam unidades em anos diferentes. Trata-se, portanto, de um tratamento escalonado em um painel longitudinal. Uma etapa futura poderá avaliar um desenho de Diferenças-em-Diferenças com tratamento escalonado, possivelmente com o estimador de Callaway–Sant'Anna, somente se os gates de identificação forem atendidos.

## 2. Fontes de dados

| Fonte | Papel |
|---|---|
| MEC/SETEC | Identificação institucional da Fase II |
| INEP/Censo Escolar | Timing e atividade das unidades |
| IBGE/DTB | Existência territorial municipal |
| IBGE/CEMPRE | Atividade econômica e outcomes |
| Cadastro nacional da Rede Federal | Exposição e elegibilidade dos controles |

In [1]:
from pathlib import Path
import sys

import plotly.graph_objects as go
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import audita_populacao_causal_cempre as d12
from visualizacao_ipt import CORES_IPT, aplicar_tema_ipt, estilizar_tabela_ipt, salvar_figura_ipt

FIGURES = ROOT / 'outputs' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
painel = d12.carrega_painel_integrado()
diagnostico = d12.auditar_fase_ii(painel)
resumo_coortes = d12.resumo_por_coorte(painel)
controles_coortes = d12.auditar_controles_por_coorte(painel)
print(f'Painel carregado offline: {len(painel):,} município-ano.')

Painel carregado offline: 72,378 município-ano.


## 3. Universo e população

A construção abaixo separa exposição observada, elegibilidade estrutural de controles e candidatos principais. Os candidatos são uma população diagnóstica: **129 não é uma amostra causal final**.

In [2]:
municipios = painel.groupby('codigo_municipio_ibge', as_index=False).first()
populacao = pd.DataFrame({
    'etapa': ['Universo municipal', 'Expostos em algum momento', 'Nunca expostos', 'Controles estruturalmente elegíveis', 'Fase II', 'Candidatos principais'],
    'n_municipios': [
        municipios['codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['ever_treated'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['sem_exposicao_observada_2007_2019'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fl_elegivel_controle_candidato'], 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['fase_ii'] == True, 'codigo_municipio_ibge'].nunique(),
        municipios.loc[municipios['candidato_amostra_principal'] == True, 'codigo_municipio_ibge'].nunique(),
    ],
})
display(estilizar_tabela_ipt(populacao))
cores_populacao = [CORES_IPT['AZUL_ESCURO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_CLARO'], CORES_IPT['CIANO'], CORES_IPT['AZUL_MEDIO'], CORES_IPT['AZUL_PRINCIPAL']]
fig = go.Figure(go.Bar(
    y=populacao['etapa'][::-1], x=populacao['n_municipios'][::-1], orientation='h',
    marker_color=cores_populacao[::-1], text=populacao['n_municipios'][::-1], textposition='outside',
    hovertemplate='<b>%{y}</b><br>%{x:,.0f} municípios<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Universo, exposição e populações diagnósticas')
fig.update_layout(showlegend=False)
fig.update_yaxes(showgrid=False)
fig.add_annotation(text='As categorias não formam um funil único', xref='paper', yref='paper', x=0, y=1.12, showarrow=False, font={'color': CORES_IPT['AZUL_MEDIO']})
salvar_figura_ipt(fig, FIGURES / '01_funil_populacao.png')
fig.show()

,etapa,n_municipios
0,Universo municipal,5570
1,Expostos em algum momento,147
2,Nunca expostos,4970
3,Controles estruturalmente elegíveis,4964
4,Fase II,147
5,Candidatos principais,129


### Interpretação

O universo contém todos os municípios observados no painel analítico. Os municípios nunca expostos não se confundem com os controles estruturais: estes últimos já incorporam regras de exclusão reproduzíveis. A população de 129 candidatos principais ainda poderá ser reduzida por decisões de identificação causal que não foram tomadas.

## 4. Painel CEMPRE

A camada técnica longa contém 506.870 linhas (5.570 municípios × 13 anos × 7 variáveis). O painel analítico contém apenas município-ano existentes territorialmente; 32 município-ano pré-criação territorial foram removidos desta camada, mas preservados na camada técnica. O outcome principal é o CEMPRE 708, pessoal ocupado assalariado.

In [3]:
resumo_painel = pd.DataFrame({
    'medida': ['Município-ano analítico', 'Municípios', 'Anos', 'Variáveis CEMPRE'],
    'valor': [len(painel), painel['codigo_municipio_ibge'].nunique(), f"{painel['ano'].min()}–{painel['ano'].max()}", 7],
})
display(estilizar_tabela_ipt(resumo_painel))

,medida,valor
0,Município-ano analítico,72378
1,Municípios,5570
2,Anos,2007–2019
3,Variáveis CEMPRE,7


### Interpretação

O painel tem cobertura longitudinal de 2007 a 2019 e preserva a unidade município-ano. A disponibilidade territorial é uma condição anterior à análise do outcome; por isso, a camada analítica não trata municípios ainda inexistentes como observações econômicas ausentes.

## 5. Coortes de tratamento

A coorte é o ano candidato de entrada em operação para cada município principal. Ela organiza o diagnóstico temporal, mas não produz por si só uma conclusão causal.

In [4]:
coortes = (diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
           .groupby('ano_coorte_candidata', as_index=False)
           .size().rename(columns={'size': 'n_candidatos'}))
coortes['ano_coorte_candidata'] = coortes['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(coortes))
fig = go.Figure(go.Bar(
    x=coortes['ano_coorte_candidata'].astype(str), y=coortes['n_candidatos'],
    marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=coortes['n_candidatos'], textposition='outside',
    hovertemplate='Coorte %{x}<br>%{y} candidatos principais<extra></extra>',
))
aplicar_tema_ipt(fig, titulo='Candidatos principais por coorte de tratamento')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '02_coortes_tratamento.png')
fig.show()

,ano_coorte_candidata,n_candidatos
0,2009,21
1,2010,27
2,2011,66
3,2012,13
4,2013,2


### Interpretação

Municípios de uma mesma coorte compartilham o mesmo ano candidato de tratamento. A maior concentração ocorre em 2011; as coortes de 2012 e 2013 são pequenas e exigirão cautela em qualquer avaliação posterior de heterogeneidade.

## 6. Linha do tempo

A linha do tempo abaixo situa as coortes dentro da janela observada. Para a coorte de 2009, apenas 2007 e 2008 estão disponíveis como anos pré-tratamento.

In [5]:
fig = go.Figure()
for _, linha in coortes.iterrows():
    rotulo = f"Coorte {linha['ano_coorte_candidata']}"
    fig.add_trace(go.Scatter(x=[2007, 2019], y=[rotulo, rotulo], mode='lines', line={'color': CORES_IPT['CINZA_GRADE'], 'width': 2}, hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=[linha['ano_coorte_candidata']], y=[rotulo], mode='markers+text', marker={'color': CORES_IPT['AZUL_PRINCIPAL'], 'size': 14}, text=[f"{linha['n_candidatos']} candidatos"], textposition='top center', hovertemplate=f"{rotulo}<br>Início: %{{x}}<br>{linha['n_candidatos']} candidatos<extra></extra>", showlegend=False))
aplicar_tema_ipt(fig, titulo='Linha do tempo das coortes de tratamento')
fig.update_xaxes(title='Ano', tickmode='linear', dtick=1, range=[2006.6, 2019.4])
fig.update_yaxes(title='', showgrid=False, categoryorder='array', categoryarray=[f'Coorte {ano}' for ano in coortes['ano_coorte_candidata'][::-1]])
fig.add_annotation(x=2008, y='Coorte 2009', text='Somente 2007 e 2008<br>antes do tratamento', showarrow=True, arrowhead=2, ax=55, ay=-45, bgcolor=CORES_IPT['CINZA_FUNDO'], bordercolor=CORES_IPT['CINZA_GRADE'], font={'color': CORES_IPT['AZUL_ESCURO']})
salvar_figura_ipt(fig, FIGURES / '03_linha_tempo_coortes.png')
fig.show()

### Interpretação

A posição da coorte no início da janela de observação limita quantos anos pré-tratamento podem ser avaliados. Isso é uma propriedade do calendário do estudo, não uma evidência de efeito nem de ausência de efeito.

## 7. Suporte temporal dos tratados

O D12 exige janelas adjacentes completas: 2 pré + 3 pós requer `g-2` a `g+2`; 3 pré + 3 pós requer `g-3` a `g+2`. Todos os anos requeridos devem existir no painel e ter CEMPRE 708 numérico utilizável.

In [6]:
suporte_tratados = resumo_coortes[['ano_coorte_candidata', 'n_candidatos', 'n_elegivel_diag_2pre_3pos', 'n_elegivel_diag_3pre_3pos']].copy()
suporte_tratados['ano_coorte_candidata'] = suporte_tratados['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(suporte_tratados))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=suporte_tratados['n_elegivel_diag_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=suporte_tratados['ano_coorte_candidata'].astype(str), y=suporte_tratados['n_elegivel_diag_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=suporte_tratados['n_elegivel_diag_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Suporte temporal dos candidatos principais')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de municípios elegíveis', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '04_suporte_temporal_tratados.png')
fig.show()

,ano_coorte_candidata,n_candidatos,n_elegivel_diag_2pre_3pos,n_elegivel_diag_3pre_3pos
0,2009,21,21,0
1,2010,27,27,27
2,2011,66,66,66
3,2012,13,13,13
4,2013,2,2,2


### Interpretação

A janela de 2 pré + 3 pós está disponível para todos os 129 candidatos. Já 3 pré + 3 pós não é possível para os 21 municípios da coorte de 2009, porque exigiria 2006, fora do painel 2007–2019. Esta limitação é de calendário, não de missing do outcome.

## 8. Disponibilidade do outcome

A tabela avalia a disponibilidade de CEMPRE 708 nas 1.677 observações dos 129 candidatos. O outcome não foi transformado e nenhum log foi aplicado.

In [7]:
candidatos = diagnostico.loc[diagnostico['candidato_amostra_principal'] == True]
disponibilidade_outcome = pd.DataFrame({
    'categoria': ['Total', 'Observado', 'Missing', 'Sigilo', 'Indisponível', 'Zero'],
    'n_observacoes': [
        candidatos['n_708_total'].sum(), candidatos['n_708_observado'].sum(), candidatos['n_708_missing'].sum(),
        candidatos['n_708_sigilo'].sum(), candidatos['n_708_indisponivel'].sum(), candidatos['n_708_zero'].sum(),
    ],
})
display(estilizar_tabela_ipt(disponibilidade_outcome))

,categoria,n_observacoes
0,Total,1677
1,Observado,1677
2,Missing,0
3,Sigilo,0
4,Indisponível,0
5,Zero,0


### Interpretação

A disponibilidade do outcome nos candidatos principais é uma verificação de qualidade de mensuração, não uma transformação analítica. Ela permite separar limitações do outcome de limitações puramente temporais do painel.

## 9. Controles por coorte

O diagnóstico aplica a mesma regra de janela adjacente aos 4.964 controles estruturalmente elegíveis. Os 4.970 municípios nunca expostos são um conjunto diferente e mais amplo.

In [8]:
controles_exibicao = controles_coortes[['ano_coorte_candidata', 'controles_2pre_3pos', 'controles_3pre_3pos']].copy()
controles_exibicao['ano_coorte_candidata'] = controles_exibicao['ano_coorte_candidata'].astype(int)
display(estilizar_tabela_ipt(controles_exibicao))
fig = go.Figure()
fig.add_bar(name='2 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_2pre_3pos'], marker_color=CORES_IPT['AZUL_PRINCIPAL'], text=controles_exibicao['controles_2pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>2 pré + 3 pós: %{y:,.0f}<extra></extra>')
fig.add_bar(name='3 pré + 3 pós', x=controles_exibicao['ano_coorte_candidata'].astype(str), y=controles_exibicao['controles_3pre_3pos'], marker_color=CORES_IPT['CIANO'], text=controles_exibicao['controles_3pre_3pos'], textposition='outside', hovertemplate='Coorte %{x}<br>3 pré + 3 pós: %{y:,.0f}<extra></extra>')
aplicar_tema_ipt(fig, titulo='Controles estruturais com janela adjacente completa')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Ano da coorte', type='category')
fig.update_yaxes(title='Número de controles', rangemode='tozero')
salvar_figura_ipt(fig, FIGURES / '05_controles_por_coorte.png')
fig.show()

,ano_coorte_candidata,controles_2pre_3pos,controles_3pre_3pos
0,2009,4964,0
1,2010,4963,4963
2,2011,4963,4963
3,2012,4963,4963
4,2013,4963,4963


### Interpretação

A única célula sigilosa do pool estrutural é o município 5003900 em 2012. Ela não afeta a coorte de 2009, cuja janela termina em 2011; afeta as coortes de 2010 a 2013, pois 2012 pertence às suas janelas adjacentes. Este diagnóstico não seleciona nem persiste uma amostra causal final.

## 10. O que já sabemos

- O painel nacional município-ano foi construído.
- O cadastro institucional foi reconstruído.
- O tratamento é escalonado.
- Há 129 candidatos principais e 4.964 controles estruturais.
- O outcome CEMPRE 708 está diagnosticado.
- O suporte temporal foi mensurado com janelas adjacentes explícitas.

## 11. O que ainda não sabemos

Ainda **não** sabemos se o desenho causal é defensável. Faltam decidir ou verificar: grupo final de comparação, antecipação, pré-tendências, comparabilidade, especificação do event study, estimador causal e robustez.

`AMOSTRA_CAUSAL_FINAL = NÃO DEFINIDA`

`DESENHO_CAUSAL_APROVADO = NÃO`

## 12. Gate de identificação causal (D13)

Os gates anteriores construíram a infraestrutura de dados: painel CEMPRE
técnico e analítico, cadastro causal institucional, integração dos dois e
auditoria do suporte temporal (D12). Nenhum desses gates avaliou se o
desenho é **defensável** para estimação causal — apenas se os dados
existem e são utilizáveis.

Este é o objetivo do **D13 — Gate de Identificação Causal**: responder, antes
de qualquer estimação, dez perguntas sobre tratamento, comparação, janela,
timing, suporte, pré-tendências e limitações. O D13 **não estima nenhum
efeito**: não roda Callaway–Sant'Anna, não calcula ATT, não faz
event-study causal, matching, propensity score, synthetic control,
regressão de efeito, TWFE causal nem bootstrap causal. É diagnóstico puro,
executado inteiramente offline sobre os artefatos já aprovados (D10/D11/D12),
sem recalculá-los.

Ao final, o notebook classifica o estado do gate — `APTO_PARA_ESPECIFICACAO`,
`REQUER_REVISAO` ou `BLOQUEADO` — mas **mesmo se apto**, mantém
`DESENHO_CAUSAL_APROVADO = NÃO` até a especificação causal ser formalmente
congelada em uma etapa futura.

## 13. O que exatamente e o tratamento?

O `CONTRATO_CAUSAL.md` define o tratamento candidato como **presenca
operacional de campus associado a Expansao Fase II no municipio** -- um
conceito institucional, nao uma leitura direta do CEMPRE. Este gate nao
redefine o tratamento a partir do comportamento do outcome economico; ele
recupera a definicao ja aprovada no cadastro causal (`D11`).

**Camadas da populacao (nao confundir):**

1. **Fase II (147 municipios)** -- populacao institucional oficial. Todos tem
   `ever_treated=true` e `pode_ser_controle=false`, sem excecao. Nem todos os
   147 sao elegiveis para a amostra causal principal.
2. **Candidato a amostra principal (129 municipios)** -- subconjunto da Fase
   II que atende, simultaneamente: status curado como `candidato_principal`
   no cadastro causal, ausencia de tratamento preexistente ao painel, uma
   `ano_coorte_candidata` definida, e elegibilidade temporal minima (2 pre +
   3 pos). Isso e uma condicao **mecanica e necessaria**, nunca suficiente --
   os 129 nao sao a amostra causal final (`CONTRATO_CAUSAL.md`, secao
   "Cadastro causal aprovado").
3. **`ano_coorte_candidata` (campo canonico da coorte)** -- o campo que
   representa a coorte candidata de cada municipio no cadastro causal. Por
   padrao (`origem_coorte='proxy_censo'`), ele e preenchido diretamente pelo
   primeiro ano observado de EPT federal ativa no Censo Escolar -- apenas uma
   **proxy anual observacional**, nao necessariamente o ano de criacao,
   autorizacao, inauguracao ou inicio das aulas. Quando existe evidencia
   institucional documentada mais forte (`origem_coorte='institucional_validada'`,
   como Sobral/CE), ela tem precedencia sobre a proxy.

**Relacao entre evento institucional, transicao e primeiro ano completo.**
O cadastro causal reserva tres campos distintos para isso --
`ano_evento_institucional`, `ano_transicao` e `primeiro_ano_completo`. A
auditoria abaixo (sem hardcode) mostra que, dos 129 candidatos principais,
**128 estao sob `origem_coorte='proxy_censo'`**: para eles, esses tres
campos vem nulos por construcao -- nao ha transicao institucional
documentada separada do primeiro ano observado no Censo, e a coorte
candidata e o proprio ano-proxy. O **unico caso** com
`origem_coorte='institucional_validada'` e **Cabo Frio/RJ (codigo
3300704)**: o IFF Cabo Frio foi inaugurado oficialmente em 05/03/2009
(fonte institucional direta do IFF), 2009 e tratado como ano parcial
(`ano_transicao=2009`, ja listado em `anos_excluir_estimacao` no cadastro
causal) e 2010 como o primeiro ano civil completo
(`primeiro_ano_completo=2010`, igual a `ano_coorte_candidata`). Este e
exatamente o caso em que o contrato ja tem uma regra inequivoca de timing --
aplicada, nao inventada aqui -- e por isso 2009 nao deve ser lido como ano
de tratamento pleno para esse municipio especifico.

In [9]:
# Confere diretamente no cadastro causal: para os 129 candidatos principais,
# os campos de transicao/evento/primeiro-ano-completo estao nulos (a coorte
# candidata E o ano-proxy do Censo, origem_coorte='proxy_censo').
candidatos_129 = painel.loc[
    (painel['fase_ii'] == True) & (painel['candidato_amostra_principal'] == True)
].groupby('codigo_municipio_ibge', as_index=False).first()

tabela_origem_coorte = candidatos_129.groupby('origem_coorte', as_index=False).agg(
    n_municipios=('codigo_municipio_ibge', 'nunique'),
    n_com_ano_transicao=('ano_transicao', lambda s: int(s.notna().sum())),
    n_com_primeiro_ano_completo=('primeiro_ano_completo', lambda s: int(s.notna().sum())),
)
display(estilizar_tabela_ipt(tabela_origem_coorte))
print(f"Municipios candidatos: {len(candidatos_129)}")
print(f"Origem da coorte candidata (unica): {sorted(candidatos_129['origem_coorte'].unique())}")

,origem_coorte,n_municipios,n_com_ano_transicao,n_com_primeiro_ano_completo
0,institucional_validada,1,1,1
1,proxy_censo,128,0,0


Municipios candidatos: 129
Origem da coorte candidata (unica): ['institucional_validada', 'proxy_censo']


## 14. Antecipacao e transicao institucional

O `CONTRATO_CAUSAL.md` (secao "Antecipacao") e o `PROTOCOLO_PRE_ANALISE.md`
(secao 6.7) sao explicitos: **a janela de antecipacao ainda nao foi
decidida**. Nao existe, nos documentos contratuais aprovados, uma regra
geral e inequivoca de quantos anos antes do primeiro ano observado (proxy
do Censo) efeitos economicos antecipados (obras, contratacao
administrativa preparatoria, expectativa de valorizacao) devem ser
excluidos da especificacao principal ou tratados como leads no
event-study.

A auditoria da secao 13 mostra que, hoje, **128 dos 129 candidatos** nao
tem `ano_transicao`/`primeiro_ano_completo` documentados individualmente
(`origem_coorte='proxy_censo'`) -- para eles nao ha, ainda, base para
diferenciar "ano de transicao" de "ano de tratamento". O **unico caso**
com essa distincao ja resolvida institucionalmente e Cabo Frio/RJ, onde o
contrato ja aplica uma regra inequivoca (2009 excluido, 2010 como coorte).
Isso e a excecao, nao a regra: nao ha uma politica geral de antecipacao
para os outros 128 candidatos, e o caso pontual de Cabo Frio nao deve ser
generalizado como se resolvesse a decisao de antecipacao para a amostra
inteira.

**Decisao pendente deste gate -- registrada, nao inventada:**

- `JANELA_ANTECIPACAO_DEFINIDA = NAO` (para a amostra em geral);
- a especificacao principal deste D13 nao cria `post` nem `event_time`;
  quando uma especificacao causal futura precisar decidir antecipacao para
  os demais candidatos, a decisao deve vir de evidencia institucional
  adicional (datas de autorizacao/obras por municipio) ou de sensibilidade
  sobre os leads do event-study -- nunca de uma regra inventada nesta etapa
  nem escolhida pelo sinal do efeito estimado (regra explicita contra
  *specification searching*, `PROTOCOLO_PRE_ANALISE.md`, secao 11).

## 15. Grupo de comparação: auditoria do pool de 4.964 controles

`fl_elegivel_controle_candidato=True` marca 4.964 municípios candidatos a
controle — nunca controles causalmente validados (`POOL_CANDIDATO_CONTROLES.md`).
A auditoria abaixo confirma empiricamente, sobre o painel integrado real,
três propriedades que o desenho precisa (sem fazer matching, sem excluir
ninguém por outcome):

In [10]:
import diagnostica_identificacao_causal as d13

COORTES_D13 = [2009, 2010, 2011, 2012, 2013]

auditoria_pool = d13.auditar_pool_controles(painel)
for chave, valor in auditoria_pool.items():
    print(f"{chave}: {valor}")

n_pool_elegivel: 4964
n_nunca_expostos: 4970
n_pool_nao_nunca_exposto: 0
codigos_pool_nao_nunca_exposto: []
n_pool_fase_ii: 0
codigos_pool_fase_ii: []
n_nunca_exposto_fora_do_pool: 6
motivos_nunca_exposto_fora_do_pool: {'universo_incompleto': 6}
pool_e_subconjunto_de_nunca_expostos: True
pool_sem_fase_ii: True


**Leitura da auditoria.** Os 4.964 elegíveis são um **subconjunto estrito**
dos 4.970 nunca expostos (`pool_e_subconjunto_de_nunca_expostos = True`) —
os dois conceitos não devem ser confundidos, e este gate não substitui
automaticamente um pelo outro. A diferença de 6 municípios (4.970 − 4.964)
não está no pool elegível porque não está presente nos 13 anos do universo
2007–2019 (`motivos_nunca_exposto_fora_do_pool = universo_incompleto`) —
são municípios criados depois de 2007 (ex.: Balneário Rincão/SC e Paraíso
das Águas/MS, ambos já documentados em D10 como criados em 2013), não uma
falha de exposição. Nenhum dos 4.964 elegíveis é município Fase II
(`pool_sem_fase_ii = True`) — nenhuma exceção institucional foi encontrada
e escondida. Portanto, para os fins deste gate:

`GRUPO_COMPARACAO_CANDIDATO = 4.964 controles estruturais elegíveis, todos
nunca expostos em 2007–2019, todos fora da Fase II` — confirmado
empiricamente, não presumido.

## 16. Janela temporal: duas especificações candidatas

Duas janelas adjacentes completas são comparadas, sem escolher pela que
preserva mais municípios:

- **Especificação candidata A — 2 pré + 3 pós**: `g-2, g-1, g, g+1, g+2`.
- **Especificação candidata B — 3 pré + 3 pós**: `g-3, g-2, g-1, g, g+1, g+2`.

A tabela e o gráfico abaixo reaproveitam a auditoria já aprovada no D12
(`resumo_coortes`, `controles_coortes`) — nenhum novo cálculo de
elegibilidade é feito aqui.

In [11]:
comparacao_janelas = resumo_coortes[[
    'ano_coorte_candidata', 'n_candidatos', 'n_elegivel_diag_2pre_3pos', 'n_elegivel_diag_3pre_3pos'
]].merge(
    controles_coortes[['ano_coorte_candidata', 'controles_2pre_3pos', 'controles_3pre_3pos']],
    on='ano_coorte_candidata',
)
comparacao_janelas['ano_coorte_candidata'] = comparacao_janelas['ano_coorte_candidata'].astype(int)
comparacao_janelas.columns = [
    'Coorte', 'N tratados', 'Tratados 2pre+3pos', 'Tratados 3pre+3pos',
    'Controles 2pre+3pos', 'Controles 3pre+3pos',
]
display(estilizar_tabela_ipt(comparacao_janelas))

total_a = int(comparacao_janelas['Tratados 2pre+3pos'].sum())
total_b = int(comparacao_janelas['Tratados 3pre+3pos'].sum())
print(f"Especificacao A (2pre+3pos): {total_a}/129 candidatos preservados.")
print(f"Especificacao B (3pre+3pos): {total_b}/129 candidatos preservados.")
print(f"Coorte 2009 sob B: {int(comparacao_janelas.loc[comparacao_janelas['Coorte'] == 2009, 'Tratados 3pre+3pos'].iloc[0])}/21 preservados.")

,Coorte,N tratados,Tratados 2pre+3pos,Tratados 3pre+3pos,Controles 2pre+3pos,Controles 3pre+3pos
0,2009,21,21,0,4964,0
1,2010,27,27,27,4963,4963
2,2011,66,66,66,4963,4963
3,2012,13,13,13,4963,4963
4,2013,2,2,2,4963,4963


Especificacao A (2pre+3pos): 129/129 candidatos preservados.
Especificacao B (3pre+3pos): 108/129 candidatos preservados.
Coorte 2009 sob B: 0/21 preservados.


**Por que a diferença não é sobre "manter mais N".** A especificação B exige
`g-3`, que para a coorte 2009 seria 2006 — fora do painel (2007–2019). Isso
é uma **limitação de calendário**, não de qualidade do outcome: os 1.677
município-ano dos 129 candidatos têm CEMPRE 708 completo (D12), então a
coorte 2009 não é excluída por dado faltante, mas por o painel simplesmente
não alcançar 2006. Adotar B como especificação principal eliminaria os 21
municípios da coorte 2009 inteiros — uma coorte real, institucionalmente
válida, não um artefato a descartar por conveniência.

## 17. Decisão provisória de janela (candidata, não congelada)

Com base exclusivamente na comparação acima — sem qualquer diagnóstico até
aqui apontando bloqueador —, a proposta deste gate é:

- **`JANELA_PRINCIPAL_CANDIDATA = 2 pré + 3 pós`** — preserva os 129/129
  candidatos e os 4.964/4.964 (coorte 2009) ou 4.963/4.963 (coortes
  2010–2013) controles estruturais, sem perder a coorte 2009 por limitação
  de calendário.
- **`JANELA_SENSIBILIDADE_CANDIDATA = 3 pré + 3 pós`** — usada como
  robustez sobre os 108/129 candidatos das coortes 2010–2013, exatamente
  porque tem mais história pré-tratamento disponível para diagnosticar
  tendências.

Isso é uma **proposta**, condicionada aos diagnósticos de pré-tendência das
seções seguintes. Se algum diagnóstico abaixo apontar evidência contrária,
esta seção deve ser revista — não confirmada por inércia.

## Identificação causal: por que Diferenças-em-Diferenças?

Comparar diretamente o outcome dos municípios tratados antes e depois do
campus chegar não isola o efeito do campus: qualquer coisa que mude ao
longo do tempo no Brasil inteiro (ciclo econômico, inflação, outra política
federal) também mudaria o outcome, mesmo sem campus nenhum.

A intuição de Diferenças-em-Diferenças (DiD) é comparar duas mudanças, não
dois níveis:

> "O quanto o tratado mudou **além** da mudança que também ocorreu no grupo
> de comparação?"

Em notação simples, com um só grupo tratado e um só grupo de controle, dois
períodos (antes/depois):

$$
DiD = \big(Y_{tratado,\,depois} - Y_{tratado,\,antes}\big) - \big(Y_{controle,\,depois} - Y_{controle,\,antes}\big)
$$

O primeiro termo (mudança do tratado) mistura o efeito do tratamento com
qualquer tendência comum. O segundo termo (mudança do controle) estima
essa tendência comum, assumindo que o controle a teria vivido igualmente
caso não fosse controle. Subtrair um do outro remove a tendência comum e
deixa (sob hipóteses, ver seção 20) o efeito atribuível ao tratamento.

**Isto é apenas intuição.** O nosso caso é mais complexo: municípios não são
tratados todos no mesmo ano — são tratados em 2009, 2010, 2011, 2012 ou
2013 (seção 19), o que a fórmula de dois grupos e dois períodos acima não
cobre diretamente.

### Tratamento escalonado: por que "tratado × pós" não basta

As coortes 2009, 2010, 2011, 2012 e 2013 (seção 5 acima) formam um
**tratamento escalonado** (*staggered adoption*): municípios diferentes são
tratados em anos diferentes, e o tratamento é absorvente — quem já foi
tratado permanece tratado nos anos seguintes.

Um modelo ingênuo de regressão com efeitos fixos de município e de ano
(TWFE — *two-way fixed effects*) e uma única variável `tratado × pós`
parece uma extensão natural do DiD de dois grupos, mas a literatura recente
(Goodman-Bacon, Callaway–Sant'Anna, Sun–Abraham, entre outros) mostrou que,
sob adoção escalonada, esse coeficiente único pode ser uma **média ponderada
com pesos negativos** de efeitos heterogêneos por coorte e por tempo desde
o tratamento — inclusive usando municípios já tratados como "controle" para
outros tratados mais tarde, contaminando a comparação. Por isso o
`CONTRATO_CAUSAL.md` já registra: **"TWFE convencional não será usado como
estimador causal principal."**

Este notebook não demonstra essa matemática — apenas registra por que ela
importa para o nosso desenho. O candidato metodológico adequado a
tratamento escalonado, mencionado no contrato e a ser estudado/implementado
em etapa futura (não aqui), é **Callaway–Sant'Anna**, que estima
$ATT(g,t)$ separadamente por coorte $g$ e tempo $t$, e só então agrega.

## A hipótese central: tendências paralelas

DiD (e suas extensões para tratamento escalonado) não exige que tratados e
controles tenham o **mesmo nível** antes do tratamento. Um município Fase II
pode ter, antes do campus chegar, um pessoal ocupado assalariado muito maior
que a mediana do pool de controles — isso por si só não invalida o desenho.

O que o DiD exige é mais sutil: que, **na ausência do tratamento**, seja
plausível que as trajetórias de tratados e controles teriam evoluído de
forma comparável (paralela) — não que partissem do mesmo lugar.

**Isso não é diretamente observável**: não existe um "mundo contrafactual
sem campus" para comparar. O que este notebook pode fazer é examinar as
trajetórias **antes** do tratamento (quando ambos os grupos ainda não foram
afetados pelo campus) como diagnóstico de plausibilidade — nunca como prova.
Pré-tendências paralelas observadas **não provam** que as tendências
pós-tratamento também seriam paralelas na ausência do tratamento; apenas
tornam essa hipótese mais ou menos plausível.

## 21. Análise de nível pré-tratamento (g-1)

Para cada coorte, a tabela compara a distribuição do CEMPRE 708 no ano
`g-1` entre os candidatos daquela coorte e o pool inteiro de 4.964/4.963
controles elegíveis (nenhum controle é excluído por outcome ou trajetória).

**Diferença de nível não reprova DiD automaticamente — é diagnóstico.** Os
municípios Fase II foram selecionados por critérios do MEC (social,
geográfico, desenvolvimentista) que, pelo DAG da seção 2 (`CONTRATO_CAUSAL.md`),
são justamente as características prévias que também afetam a trajetória
econômica — por isso é esperado, não surpreendente, que tratados e
controles tenham níveis de baseline muito diferentes.

In [12]:
baseline_niveis = d13.baseline_pre_tratamento_por_coorte(painel)
baseline_exibicao = baseline_niveis.copy()
baseline_exibicao['ano_coorte_candidata'] = baseline_exibicao['ano_coorte_candidata'].astype(int)
for col in ['media_tratados', 'mediana_tratados', 'p25_tratados', 'p75_tratados',
            'media_controles', 'mediana_controles', 'p25_controles', 'p75_controles']:
    baseline_exibicao[col] = baseline_exibicao[col].round(1)
display(estilizar_tabela_ipt(baseline_exibicao))

,ano_coorte_candidata,ano_baseline_g_menos_1,n_tratados,n_controles,media_tratados,mediana_tratados,p25_tratados,p75_tratados,media_controles,mediana_controles,p25_controles,p75_controles
0,2009,2008,21,4964,20236.300000,14970.000000,4887.000000,29158.000000,2321.900000,644.000000,315.000000,1611.800000
1,2010,2009,27,4964,12689.300000,7825.000000,2578.500000,19570.000000,2449.300000,701.500000,350.000000,1720.200000
2,2011,2010,66,4964,17514.500000,9599.500000,3574.200000,17588.800000,2641.800000,745.500000,367.000000,1831.500000
3,2012,2011,13,4964,30395.800000,21158.000000,9120.000000,37226.000000,2763.700000,789.000000,390.000000,1928.500000
4,2013,2012,2,4963,27015.500000,27015.500000,19541.200000,34489.800000,2799.900000,763.000000,373.000000,1935.000000


### 06 — Distribuição do outcome no baseline (g-1), tratados x controles

Boxplot com eixo Y logarítmico **apenas como recurso visual** (os dados
não são transformados para nenhuma análise — só a escala do eixo do
gráfico muda, para tornar visível a distribuição do pool de controles, que
vive numa escala muito menor que a dos tratados). Nenhum outlier é
removido.

In [13]:
dist_longa = d13.distribuicao_baseline_longa(painel)
dist_longa['coorte_rotulo'] = 'Coorte ' + dist_longa['ano_coorte_candidata'].astype(int).astype(str)

fig = go.Figure()
for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
    sub = dist_longa.loc[dist_longa['grupo'] == grupo]
    fig.add_trace(go.Box(
        x=sub['coorte_rotulo'], y=sub['pessoal_ocupado_assalariado'], name=grupo,
        marker_color=cor, boxpoints='outliers',
    ))
aplicar_tema_ipt(fig, titulo='Distribuicao do CEMPRE 708 no baseline (g-1) por coorte')
fig.update_layout(boxmode='group')
fig.update_xaxes(title='Coorte', categoryorder='array', categoryarray=[f'Coorte {a}' for a in sorted(dist_longa['ano_coorte_candidata'].unique())])
fig.update_yaxes(title='Pessoal ocupado assalariado (g-1, escala log)', type='log')
fig.add_annotation(text='Eixo log apenas para visualizacao; dado nao transformado na analise', xref='paper', yref='paper', x=0, y=1.1, showarrow=False, font={'color': CORES_IPT['AZUL_MEDIO'], 'size': 11})
salvar_figura_ipt(fig, FIGURES / '06_distribuicao_baseline_tratados_controles.png')
fig.show()

## 23. Pré-tendências descritivas em níveis (somente anos pré-tratamento)

Para cada coorte, a mediana do CEMPRE 708 dos candidatos daquela coorte e
do pool de controles, ano a ano, usando **exclusivamente anos anteriores a
`g`** — nenhuma observação pós-tratamento entra nesta série. Cada coorte
usa toda a história pré disponível no painel (não trava em 2 anos): a
coorte 2013 mostra 2007–2012 (6 anos), a 2009 mostra só 2007–2008 (2 anos).

Este é um diagnóstico de plausibilidade, não uma afirmação causal.

In [14]:
series_niveis = pd.concat(
    [d13.serie_pre_tendencia_niveis(painel, coorte) for coorte in COORTES_D13],
    ignore_index=True,
)
n_pre_por_coorte = {c: int(c - 2007) for c in COORTES_D13}
print('Anos pre disponiveis por coorte (nao limitado a 2):')
for c, n in n_pre_por_coorte.items():
    print(f'  Coorte {c}: {n} anos pre ({2007}-{c - 1})')

from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=3, subplot_titles=[f'Coorte {c}' for c in COORTES_D13], shared_yaxes=False)
posicoes = [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2)]
for (coorte, (linha, coluna)) in zip(COORTES_D13, posicoes):
    sub = series_niveis.loc[series_niveis['ano_coorte_candidata'] == coorte]
    for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
        sg = sub.loc[sub['grupo'] == grupo].sort_values('ano')
        fig.add_trace(
            go.Scatter(x=sg['ano'], y=sg['mediana_708'], mode='lines+markers', name=grupo,
                       line={'color': cor}, marker={'color': cor}, showlegend=(linha, coluna) == (1, 1)),
            row=linha, col=coluna,
        )
aplicar_tema_ipt(fig, titulo='Pre-tendencias em niveis (mediana CEMPRE 708) -- somente pre-tratamento')
fig.update_xaxes(dtick=1)
fig.update_layout(height=650)
salvar_figura_ipt(fig, FIGURES / '07_pre_tendencias_niveis.png')
fig.show()

Anos pre disponiveis por coorte (nao limitado a 2):
  Coorte 2009: 2 anos pre (2007-2008)
  Coorte 2010: 3 anos pre (2007-2009)
  Coorte 2011: 4 anos pre (2007-2010)
  Coorte 2012: 5 anos pre (2007-2011)
  Coorte 2013: 6 anos pre (2007-2012)


## 24. Pré-tendências normalizadas (índice, g-1 = 100)

Como tratados e controles têm níveis de baseline muito diferentes (seções
21-22), a comparação de **trajetória relativa** usa um índice: para cada
coorte e grupo, `g-1 = 100`, usando a mediana do grupo como base — **apenas
para visualização**, nunca substituindo o outcome bruto usado por qualquer
estimador futuro. Antes de normalizar, a rotina confirma que a mediana em
`g-1` é positiva em cada caso; se não fosse, o índice não seria calculado
(nenhum tratamento seria inventado).

In [15]:
indices = pd.concat(
    [d13.indice_pre_tendencia_normalizado(d13.serie_pre_tendencia_niveis(painel, c), coorte=c) for c in COORTES_D13],
    ignore_index=True,
)
indices_validos = indices.dropna(subset=['indice_708'])
print(f"Linhas com indice calculado: {len(indices_validos)}/{len(indices)} (mediana g-1 > 0 confirmada em todos os casos calculados).")

fig = make_subplots(rows=2, cols=3, subplot_titles=[f'Coorte {c}' for c in COORTES_D13], shared_yaxes=True)
for (coorte, (linha, coluna)) in zip(COORTES_D13, posicoes):
    sub = indices.loc[(indices['ano_coorte_candidata'] == coorte) & (indices['ano'] < coorte)]
    for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
        sg = sub.loc[sub['grupo'] == grupo].sort_values('ano')
        fig.add_trace(
            go.Scatter(x=sg['ano'], y=sg['indice_708'], mode='lines+markers', name=grupo,
                       line={'color': cor}, marker={'color': cor}, showlegend=(linha, coluna) == (1, 1)),
            row=linha, col=coluna,
        )
    fig.add_hline(y=100, line={'color': CORES_IPT['CINZA_GRADE'], 'dash': 'dot'}, row=linha, col=coluna)
aplicar_tema_ipt(fig, titulo='Indice de pre-tendencia (g-1=100) -- somente k<0, apenas visualizacao')
fig.update_xaxes(dtick=1)
fig.update_yaxes(title='Indice CEMPRE 708 (g-1=100)')
fig.update_layout(height=650)
salvar_figura_ipt(fig, FIGURES / '08_pre_tendencias_indice.png')
fig.show()

Linhas com indice calculado: 40/40 (mediana g-1 > 0 confirmada em todos os casos calculados).


## 26. Mudança pré-tratamento: `g-2` para `g-1`

Variação descritiva (absoluta e percentual) da mediana do CEMPRE 708 entre
os dois últimos anos antes do tratamento, tratados vs. controles. **Não é
um teste formal de tendências paralelas** — apenas mais um número
descritivo de apoio ao diagnóstico.

In [16]:
mudanca = d13.mudanca_pre_g2_g1(painel)
mudanca_exibicao = mudanca.copy()
mudanca_exibicao['variacao_absoluta'] = mudanca_exibicao['variacao_absoluta'].round(1)
mudanca_exibicao['variacao_percentual'] = mudanca_exibicao['variacao_percentual'].round(2)
display(estilizar_tabela_ipt(mudanca_exibicao))

fig = go.Figure()
for grupo, cor in [('tratados', CORES_IPT['AZUL_PRINCIPAL']), ('controles', CORES_IPT['CIANO'])]:
    sub = mudanca.loc[mudanca['grupo'] == grupo]
    fig.add_bar(name=grupo, x=sub['ano_coorte_candidata'].astype(int).astype(str), y=sub['variacao_percentual'],
                marker_color=cor, text=sub['variacao_percentual'].round(1), textposition='outside')
aplicar_tema_ipt(fig, titulo='Variacao percentual do CEMPRE 708 mediano entre g-2 e g-1')
fig.update_layout(barmode='group')
fig.update_xaxes(title='Coorte', type='category')
fig.update_yaxes(title='Variacao percentual (%)')
salvar_figura_ipt(fig, FIGURES / '09_mudanca_pre_g2_g1.png')
fig.show()

,ano_coorte_candidata,grupo,ano_g2,ano_g1,mediana_g2,mediana_g1,variacao_absoluta,variacao_percentual
0,2009,tratados,2007,2008,14627.000000,14970.000000,343.000000,2.340000
1,2009,controles,2007,2008,638.000000,644.000000,6.000000,0.940000
2,2010,tratados,2008,2009,7520.000000,7825.000000,305.000000,4.060000
3,2010,controles,2008,2009,644.000000,701.500000,57.500000,8.930000
4,2011,tratados,2009,2010,8444.000000,9599.500000,1155.500000,13.680000
5,2011,controles,2009,2010,701.500000,745.500000,44.000000,6.270000
6,2012,tratados,2010,2011,20428.000000,21158.000000,730.000000,3.570000
7,2012,controles,2010,2011,745.500000,789.000000,43.500000,5.840000
8,2013,tratados,2011,2012,27315.500000,27015.500000,-300.000000,-1.100000
9,2013,controles,2011,2012,789.000000,763.000000,-26.000000,-3.300000


### Limitação da coorte 2009

O painel começa em 2007. A coorte 2009 possui apenas **2007 e 2008** como
anos pré-tratamento — não há como observar 2006. Isso significa que existe
apenas **uma** mudança pré observável: `2007 → 2008` (mostrada acima:
+2,3% para os tratados, +0,9% para os controles). Essa é uma limitação
severa de capacidade diagnóstica — com um único ponto de variação, não é
possível avaliar se a trajetória pré-tratamento é estável ao longo de vários
anos, só entre dois.

Isso **não** é motivo para excluir a coorte 2009 automaticamente:

- a especificação principal candidata (2 pré + 3 pós, seção 17) **inclui**
  a coorte 2009 — ela tem suporte temporal completo (21/21) sob essa janela;
- a especificação de sensibilidade (3 pré + 3 pós) **não pode** incluí-la —
  exigiria 2006, fora do painel (0/21 sob essa janela, seção 16).

### Coortes 2010–2013: mais história pré disponível

Diferente da coorte 2009, as coortes seguintes têm mais anos pré
disponíveis no painel — e as figuras 07/08 acima já usam essa história
inteira (não travada em 2 anos), porque a janela `2pre+3pos` é a
especificação **candidata**, não o limite dos dados observáveis para
diagnóstico:

In [17]:
print('Anos pre efetivamente disponiveis no painel, por coorte (para diagnostico):')
for c in COORTES_D13:
    print(f'  Coorte {c}: {c - 2007} anos pre ({2007}-{c - 1})')

Anos pre efetivamente disponiveis no painel, por coorte (para diagnostico):
  Coorte 2009: 2 anos pre (2007-2008)
  Coorte 2010: 3 anos pre (2007-2009)
  Coorte 2011: 4 anos pre (2007-2010)
  Coorte 2012: 5 anos pre (2007-2011)
  Coorte 2013: 6 anos pre (2007-2012)


## 29. O que esta etapa não faz com os controles

Este gate audita e descreve o pool de 4.964 controles estruturais, mas
**não** faz nenhuma das seguintes coisas:

- matching (par a par ou por propensity score);
- exclusão de controle por nível de outcome diferente do dos tratados;
- *trimming* baseado em outcome;
- seleção individual de controle olhando sua trajetória.

Todas as tabelas e figuras acima usam o pool institucional canônico
inteiro (4.964, ou 4.963 quando a janela adjacente inclui 2012 — a única
célula sigilosa do pool, ver D12). Isso é deliberado: balanceamento e
seleção de controles pertencem a uma etapa posterior, condicionada a este
gate ter sido considerado apto.

## 30. Spillover e arranjos populacionais (diagnósticos já existentes)

`DIAGNOSTICO_SPILLOVER_FASE_II.md` e `DIAGNOSTICO_ARRANJOS_POPULACIONAIS_FASE_II.md`
já produziram, em etapas anteriores, uma base quantitativa de proximidade
geográfica (distância de Haversine entre sedes municipais) e de arranjos
populacionais entre os 4.964 candidatos a controle e os 147 municípios Fase
II. Ambos os documentos são explícitos: essas informações são
**diagnósticos**, não critérios automáticos de exclusão — o
`CONTRATO_CAUSAL.md` (seção "Spillovers") registra que nenhum raio de
distância é fixado sem justificativa substantiva (dados de deslocamento
pendular) ou análise de sensibilidade.

Este D13 não reabre essa análise espacial. Fica registrado como **ressalva
para robustez futura**: se a especificação causal for congelada, os
diagnósticos de spillover/arranjos populacionais já produzidos devem ser
revisitados para decidir se algum critério geográfico de exclusão é
justificável — não antes disso, e não automaticamente.

## 31. Tabela de gate por coorte

Combina, por coorte, o suporte temporal já auditado no D12 com uma
categoria **objetiva e descritiva** de disponibilidade de diagnóstico de
pré-tendência (nunca "boa"/"ruim" — apenas quantos períodos pré existem).

In [18]:
tabela_gate = d13.tabela_gate_por_coorte(painel)
display(estilizar_tabela_ipt(tabela_gate))

,ano_coorte_candidata,n_tratados,n_pre_disponiveis,suporte_2pre3pos,suporte_3pre3pos,controles_2pre3pos,controles_3pre3pos,observacao_pre_tendencia
0,2009,21,2,21,0,4964,0,LIMITADA_2_PERIODOS_PRE
1,2010,27,3,27,27,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
2,2011,66,4,66,66,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
3,2012,13,5,13,13,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL
4,2013,2,6,2,2,4963,4963,DIAGNOSTICO_PRE_DISPONIVEL


## 32. Gate de identificação causal — classificação

Este gate **não** declara `DESENHO_CAUSAL_APROVADO = SIM` — essa decisão
pertence a uma etapa futura, depois de a especificação causal ser
formalmente congelada. O que este gate faz é classificar o estado atual em
uma de três categorias, a partir exclusivamente dos diagnósticos executados
acima:

- **`APTO_PARA_ESPECIFICACAO`** — população coerente, controles canônicos
  coerentes, timing definido (dentro do que o cadastro aprovado permite),
  nenhum bloqueador objetivo apareceu nos diagnósticos, e as ressalvas estão
  explicitadas;
- **`REQUER_REVISAO`** — há achado que precisa de decisão antes de seguir;
- **`BLOQUEADO`** — incompatibilidade estrutural grave.

**Checagem programática (sem hardcode) dos critérios de bloqueio objetivo:**

In [19]:
bloqueadores = []

if not auditoria_pool['pool_e_subconjunto_de_nunca_expostos']:
    bloqueadores.append('Pool de controles contem municipio com exposicao observada.')
if not auditoria_pool['pool_sem_fase_ii']:
    bloqueadores.append('Pool de controles contem municipio Fase II.')
if int(tabela_gate['suporte_2pre3pos'].sum()) != int(tabela_gate['n_tratados'].sum()):
    bloqueadores.append('Especificacao candidata principal (2pre+3pos) nao cobre todos os 129 candidatos.')
if int((tabela_gate['controles_2pre3pos'] == 0).sum()) > 0:
    bloqueadores.append('Alguma coorte ficou sem nenhum controle elegivel sob 2pre+3pos.')

ressalvas = [
    'Antecipacao (janela de leads) ainda NAO definida -- CONTRATO_CAUSAL.md, secao Antecipacao (pendencia registrada na secao 14).',
    'Grupo de comparacao final (never-treated vs. not-yet-treated, exclusoes adicionais por spillover) ainda em aberto -- CONTRATO_CAUSAL.md, secao Grupo de comparacao candidato.',
    'Spillover/arranjos populacionais permanecem apenas diagnosticos, nao aplicados como exclusao (secao 30).',
    'Coorte 2009 tem apenas uma mudanca pre observavel (2007->2008); nao suporta 3pre+3pos (secao 27).',
    'Uma unica celula de CEMPRE 708 sigilosa no pool estrutural (municipio 5003900, ano 2012) reduz o pool de 4964 para 4963 nas coortes 2010-2013 (D12).',
    'Baseline em nivel muito diferente entre tratados e controles (secao 21) -- diagnostico esperado pelo DAG (selecao nao aleatoria via criterios MEC), nao reprova o desenho por si so.',
]

GATE_IDENTIFICACAO = 'BLOQUEADO' if bloqueadores else 'APTO_PARA_ESPECIFICACAO'

DESENHO_CAUSAL_APROVADO = 'NAO'

print(f'GATE_IDENTIFICACAO = {GATE_IDENTIFICACAO}')
print(f'DESENHO_CAUSAL_APROVADO = {DESENHO_CAUSAL_APROVADO}')
print()
print(f'Bloqueadores objetivos encontrados: {len(bloqueadores)}')
for b in bloqueadores:
    print(f'  - {b}')
print()
print(f'Ressalvas explicitadas: {len(ressalvas)}')
for r in ressalvas:
    print(f'  - {r}')

GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO
DESENHO_CAUSAL_APROVADO = NAO

Bloqueadores objetivos encontrados: 0

Ressalvas explicitadas: 6
  - Antecipacao (janela de leads) ainda NAO definida -- CONTRATO_CAUSAL.md, secao Antecipacao (pendencia registrada na secao 14).
  - Grupo de comparacao final (never-treated vs. not-yet-treated, exclusoes adicionais por spillover) ainda em aberto -- CONTRATO_CAUSAL.md, secao Grupo de comparacao candidato.
  - Spillover/arranjos populacionais permanecem apenas diagnosticos, nao aplicados como exclusao (secao 30).
  - Coorte 2009 tem apenas uma mudanca pre observavel (2007->2008); nao suporta 3pre+3pos (secao 27).
  - Uma unica celula de CEMPRE 708 sigilosa no pool estrutural (municipio 5003900, ano 2012) reduz o pool de 4964 para 4963 nas coortes 2010-2013 (D12).
  - Baseline em nivel muito diferente entre tratados e controles (secao 21) -- diagnostico esperado pelo DAG (selecao nao aleatoria via criterios MEC), nao reprova o desenho por si so.


**Resultado: `GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO`**, com as
ressalvas listadas acima explicitamente carregadas para a próxima etapa.
Nenhum bloqueador objetivo apareceu: o pool de controles é limpo (subconjunto
de nunca expostos, sem Fase II), a especificação candidata principal
(2 pré + 3 pós) cobre 129/129 candidatos, e todas as coortes têm pelo menos
um controle elegível. `DESENHO_CAUSAL_APROVADO` permanece `NÃO` — este gate
autoriza avançar para o **desenho** da especificação, não para a
estimação.

## 33. Propostas candidatas (não congeladas)

A partir dos diagnósticos acima, as seguintes propostas ficam registradas
para a etapa de especificação — como **propostas**, não fatos consumados:

| Elemento | Proposta candidata |
|---|---|
| Grupo de comparação | 4.964 controles estruturais elegíveis, nunca tratados/expostos em 2007–2019, confirmados sem Fase II (seção 15) |
| Janela principal | 2 pré + 3 pós (seção 17) |
| Janela de sensibilidade | 3 pré + 3 pós, restrita às coortes 2010–2013 (seção 17) |
| Outcome principal | CEMPRE 708 (pessoal ocupado assalariado), em nível, sem transformação |
| Estimador futuro | Callaway–Sant'Anna / DiD para tratamento escalonado (não implementado aqui) |

Nenhum estimador foi executado. Nenhum efeito foi calculado.

## 34. O que este gate confirma e o que ainda falta

**Confirmado neste D13:**

- o tratamento é institucional (proxy do Censo), não redefinido pelo CEMPRE;
- o pool de 4.964 controles é limpo (nunca expostos, sem Fase II);
- a especificação 2 pré + 3 pós preserva os 129 candidatos e a coorte 2009;
- o outcome tem suporte completo nas janelas candidatas (herdado do D12);
- pré-tendências em nível e em índice foram diagnosticadas, sem contaminação
  por dados pós-tratamento;
- a coorte 2009 tem uma limitação de diagnóstico reconhecida e registrada,
  não escondida.

**Ainda em aberto (não resolvido aqui, de propósito):**

- janela de antecipação;
- escolha final never-treated vs. not-yet-treated e exclusões adicionais de
  spillover;
- covariáveis de balanceamento fundamentadas por literatura;
- estratégia de inferência/clusterização;
- a especificação causal formal (ainda não congelada).

`AMOSTRA_CAUSAL_FINAL = NÃO DEFINIDA`

`DESENHO_CAUSAL_APROVADO = NÃO`

`GATE_IDENTIFICACAO = APTO_PARA_ESPECIFICACAO`

### Próxima etapa

Com o gate de identificação classificado como apto, o próximo passo é
**formalizar e congelar a especificação causal** (janela definitiva,
regras de antecipação e spillover, covariáveis de balanceamento) e só
então avaliar suporte comum/balanceamento e executar event-study/ATT.
Nenhuma dessas etapas é executada neste notebook.